# Notebook 04: Controlled Drift Injection (RQ3)

**RQ3:** How do different drift types affect fairness metrics?

Factorial design: 3 types x 3 severities x 3 metrics x 3 models x 3 tasks x 30 reps = 7,290 runs


In [ ]:
import sys, os, pickle
if 'google.colab' in sys.modules:
    sys.path.insert(0, '/content/FairDrift-code')
else:
    sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from joblib import Parallel, delayed

from src import config
from src.drift_injector import DriftInjector, run_single_injection_experiment

with open('../outputs/checkpoint_01.pkl', 'rb') as f:
    ckpt01 = pickle.load(f)
with open('../outputs/checkpoint_02.pkl', 'rb') as f:
    ckpt02 = pickle.load(f)

windows = ckpt01['windows']
feature_cols = ckpt01['feature_cols']
models = ckpt02['models']
scaler = ckpt02['scaler']
print('Ready for injection experiments')


## 1. Run Factorial Experiment

In [ ]:
# Use W3 as the deployment window to inject drift into
W3 = windows[2].copy()

drift_types = ['covariate', 'label', 'concept']
severity_levels = [0, 1, 2]  # low, medium, high

all_results = []
total = len(drift_types) * len(severity_levels) * len(models) * config.N_REPLICATIONS
print(f'Total experiments: {total}')

for rep in tqdm(range(config.N_REPLICATIONS), desc='Replications'):
    seed = config.RANDOM_SEEDS[rep % len(config.RANDOM_SEEDS)] + rep
    
    for drift_type in drift_types:
        for severity in severity_levels:
            for (model_type, task), model in models.items():
                # Prepare data
                test_df = W3.copy()
                
                # Need to use scaled features for LR/MLP
                # For injection, we work on raw features then scale for prediction
                result = run_single_injection_experiment(
                    df=test_df,
                    model=model,
                    feature_cols=feature_cols,
                    task=task,
                    protected_col='race',
                    target_subgroup='AfricanAmerican',
                    reference_group='Caucasian',
                    drift_type=drift_type,
                    severity_index=severity,
                    seed=seed,
                )
                result['model'] = model_type
                result['task'] = task
                result['replication'] = rep
                all_results.append(result)

rq3_df = pd.DataFrame(all_results)
print(f'\nCompleted: {len(rq3_df)} experimental conditions')
rq3_df.head()


## 2. Three-Way ANOVA (drift_type x severity x metric)

In [ ]:
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Reshape for ANOVA: each row = one metric observation
melted = rq3_df.melt(
    id_vars=['drift_type', 'severity', 'model', 'task', 'replication'],
    value_vars=['eod', 'dpd', 'ece'],
    var_name='metric', value_name='value'
).dropna()

print('DRIFT TYPE x METRIC INTERACTION')
print('=' * 60)

# Mean values per drift_type x metric
summary = melted.groupby(['drift_type', 'metric'])['value'].agg(['mean', 'std', 'count']).reset_index()
pivot = summary.pivot(index='drift_type', columns='metric', values='mean')
print(pivot.round(4))

# Formal ANOVA per metric
print('\nANOVA: Effect of drift type on each metric')
for metric in ['eod', 'dpd', 'ece']:
    groups = [melted[(melted['metric']==metric) & (melted['drift_type']==dt)]['value'].values
              for dt in drift_types]
    f_stat, p_val = f_oneway(*groups)
    print(f'  {metric.upper()}: F={f_stat:.2f}, p={p_val:.2e}')


In [ ]:
# Figure: Heatmap of drift-type x metric response
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='RdYlGn_r', ax=ax, linewidths=0.5)
ax.set_title('Mean Fairness Metric Response by Drift Type')
ax.set_ylabel('Drift Type')
ax.set_xlabel('Fairness Metric')
plt.tight_layout()
plt.savefig('../outputs/figures/fig05_drift_type_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


## 3. Save RQ3 Results

In [ ]:
rq3_df.to_csv('../outputs/metrics/rq3_factorial_results.csv', index=False)
checkpoint_04 = {'rq3_df': rq3_df}
with open('../outputs/checkpoint_04.pkl', 'wb') as f:
    pickle.dump(checkpoint_04, f)
print('Saved: checkpoint_04.pkl + rq3_factorial_results.csv')
